# Lesson 01 - AI एजेंट्स का परिचय

**AI एजेंट्स फॉर बिगिनर्स** कोर्स के पहले पाठ में आपका स्वागत है!

एक **AI एजेंट** एक ऐसा प्रोग्राम है जो एक बड़े भाषा मॉडल (LLM) का उपयोग अपने तर्कशक्ति इंजन के रूप में करता है और वास्तविक दुनिया में *क्रियाएं* कर सकता है — API कॉल करना, डेटाबेस क्वेरी करना, या कोड चलाना — ताकि उपयोगकर्ता की ओर से एक लक्ष्य प्राप्त कर सके।

इस नोटबुक में आप अपना पहला एजेंट बनाएंगे: एक **ट्रैवल एजेंट** जो छुट्टियों के लिए स्थान सुझाता है। इस प्रक्रिया में आप सीखेंगे कि कैसे:

1. **Microsoft Agent Framework** का उपयोग करके Azure AI Foundry Agent सेवा से कनेक्ट करें।
2. एजेंट को एक **टूल** दें — एक सामान्य Python फ़ंक्शन जो वह कॉल कर सकता है।
3. एजेंट चलाएं और इसके जवाब की जांच करें।
4. एजेंट के जवाब को टोकन-दर-टोकन स्ट्रीम करें।


## सेटअप

इस नोटबुक को चलाने से पहले, सुनिश्चित करें कि आपके पास निम्नलिखित हैं:

1. **एक Azure AI Foundry परियोजना** जिसमें एक तैनात चैट मॉडल हो (जैसे `gpt-4o-mini`)।
2. **Azure CLI के साथ लॉग इन किया हुआ हो** — अपने टर्मिनल में `az login` चलाएं।
3. **आवश्यक पर्यावरण चर सेट किए हुए हों:**
   - `AZURE_AI_PROJECT_ENDPOINT` — आपकी Azure AI Foundry परियोजना का एंडपॉइंट।
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — आपके तैनात मॉडल का नाम।

नीचे वाली सेल में आवश्यक पायथन पैकेज स्थापित किए गए हैं।


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## अपना पहला एजेंट बनाना

एक एजेंट को दो चीजें चाहिए:

- **निर्देश** जो उसे बताएं *वह कौन है* और *कैसे व्यवहार करना है* (एक सिस्टम प्रॉम्प्ट)।
- **टूल्स** — Python फ़ंक्शन जिन्हें `@tool` से सजाया गया है जिन्हें एजेंट जानकारी प्राप्त करने या क्रियाएं करने के लिए कॉल कर सकता है।

नीचे हमने एक सरल टूल परिभाषित किया है जो लोकप्रिय छुट्टी स्थलों की सूची लौटाता है। जब कोई उपयोगकर्ता यात्रा सिफारिशें पूछेगा तो एजेंट इस टूल का उपयोग करेगा।


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## स्ट्रीमिंग प्रतिक्रियाएँ

एक अधिक इंटरएक्टिव अनुभव के लिए आप एजेंट की प्रतिक्रिया **स्ट्रीम** कर सकते हैं। पूरी प्रतिक्रिया की प्रतीक्षा करने के बजाय, एजेंट जैसे-जैसे टेक्स्ट उत्पन्न होता है, उसे हिस्सों में देता है। यह खासकर चैट इंटरफेस में उपयोगी होता है जहाँ आप आउटपुट को वास्तविक समय में दिखाना चाहते हैं।


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## सारांश

इस पाठ में आपने सीखा कि कैसे:

- **एक प्रोवाइडर बनाएँ** जो `AzureAIProjectAgentProvider` के माध्यम से Azure AI Foundry Agent Service से जुड़ता है।
- **एक टूल परिभाषित करें** `@tool` डेकोरेटर का उपयोग करके ताकि एजेंट आपके Python फ़ंक्शंस को कॉल कर सके।
- **एजेंट चलाएँ** एक उपयोगकर्ता संदेश के साथ और इसकी प्रतिक्रिया प्रिंट करें।
- **रियल-टाइम आउटपुट के लिए प्रतिक्रियाओं को स्ट्रीम करें**।

अगले पाठ में हम एजेंटिक फ्रेमवर्क्स की गहराई से जांच करेंगे और सीखेंगे कि एजेंटों को अधिक शक्तिशाली टूल और बहु-चरण तर्क क्षमताएं कैसे दें।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:  
इस दस्तावेज़ का अनुवाद एआई अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयासरत हैं, कृपया ध्यान दें कि स्वचालित अनुवाद में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानवीय अनुवाद की सलाह दी जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम जिम्मेदार नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
